<a href="https://colab.research.google.com/github/detektor777/colab_list_image/blob/main/mambair_super_resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title ##**Install** { display-mode: "form" }
import ctypes, glob, os

!git clone --depth 1 https://github.com/csguoh/MambaIR.git
%cd MambaIR

!pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

!pip install -q https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/causal_conv1d-1.5.0.post8+cu12torch2.6cxx11abiFALSE-cp312-cp312-linux_x86_64.whl
!pip install -q https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4+cu12torch2.6cxx11abiFALSE-cp312-cp312-linux_x86_64.whl --no-deps

# mamba_ssm relies on older generation-API classes that were removed in transformers 5.0,
# so we pin a version below 5.0 where they still exist.
!pip install -q "transformers<5.0"

# Download the MambaIRv2 x4 super-resolution model weights
!wget -q https://huggingface.co/cguoh/MambaIR/resolve/main/MambaIRv1_ckpt/MambaIR_real.pth -P ./experiments

!pip install -q nvidia-cuda-runtime-cu11==11.8.89
cudart_dir = glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cuda_runtime/lib')[0]
matches = glob.glob(os.path.join(cudart_dir, 'libcudart.so.11.0*'))
if matches:
    ctypes.CDLL(matches[0], mode=ctypes.RTLD_GLOBAL)
    os.environ['LD_LIBRARY_PATH'] = cudart_dir + ':' + os.environ.get('LD_LIBRARY_PATH', '')
    print('libcudart preloaded:', matches[0])
else:
    print('cu11 runtime not needed')

import torch
print('torch:', torch.__version__, '| cxx11 abi:', torch.compiled_with_cxx11_abi())

In [ ]:
#@title ##**Upload Images** { display-mode: "form" }
import os
import glob
import shutil
from google.colab import files

upload_folder = './samples/inputs'
result_folder = './samples/results'

if os.path.isdir(upload_folder):
    shutil.rmtree(upload_folder)
if os.path.isdir(result_folder):
    shutil.rmtree(result_folder)

os.makedirs(upload_folder, exist_ok=True)
os.makedirs(result_folder, exist_ok=True)

# Upload images — you can select multiple files at once
uploaded = files.upload()
for filename in uploaded.keys():
    dst_path = os.path.join(upload_folder, filename)
    print(f'Moved {filename} -> {dst_path}')
    shutil.move(filename, dst_path)

print(f'\n{len(uploaded)} image(s) uploaded to {upload_folder}')

In [ ]:
#@title ##**Run** { display-mode: "form" }
import torch, os, gc, cv2, sys
from tqdm import tqdm
from basicsr.archs.mambair_arch import MambaIR
from basicsr.utils import img2tensor, tensor2img

def gpu_mem(tag=""):
    a = torch.cuda.memory_allocated() / 1024**3
    r = torch.cuda.memory_reserved() / 1024**3
    print(f"[{tag}] allocated={a:.2f} GiB reserved={r:.2f} GiB")

for name in ("model", "img", "output"):
    globals().pop(name, None)
sys.last_traceback = None
gc.collect()
torch.cuda.empty_cache()
gpu_mem("start")

model = MambaIR(
    upscale=4,
    in_chans=3,
    img_size=64,
    img_range=1.,
    d_state=16,
    depths=[6, 6, 6, 6, 6, 6],
    embed_dim=180,
    mlp_ratio=2,
    upsampler='pixelshuffle',
    resi_connection='1conv',
).cuda().eval()

ckpt = torch.load('./experiments/MambaIR_real.pth', weights_only=False)
state_dict = ckpt['params'] if 'params' in ckpt else ckpt
model.load_state_dict(state_dict, strict=True)
gpu_mem("after model load")

for file_name in tqdm(os.listdir(upload_folder)):
    file_path = os.path.join(upload_folder, file_name)
    img = None
    output = None
    try:
        img = cv2.imread(file_path) / 255.
        img = img2tensor(img, bgr2rgb=True, float32=True).unsqueeze(0).cuda()

        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
            output = model(img)

        output = tensor2img([output.detach().cpu().float()])
        save_path = os.path.join(result_folder, file_name)
        cv2.imwrite(save_path, output)
        gpu_mem(file_name)

    finally:
        del img, output
        sys.last_traceback = None
        gc.collect()
        torch.cuda.empty_cache()

print(f'Done! Processed {len(os.listdir(upload_folder))} image(s) -> {result_folder}')
gpu_mem("final")

In [ ]:
#@title ##**Visualization** { display-mode: "form" }
import os
import base64
from io import BytesIO
import PIL.Image
from IPython.display import HTML, display

def is_image_file(filename):
    image_extensions = {'.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff'}
    return os.path.splitext(filename.lower())[1] in image_extensions

def resize_image_maintain_aspect(image, max_width, target_height=None):
    width, height = image.size
    if width > max_width:
        new_height = int(height * max_width / width)
        image = image.resize((max_width, new_height), PIL.Image.Resampling.LANCZOS)
    if target_height is not None and image.size[1] != target_height:
        new_width = int(image.size[0] * target_height / image.size[1])
        image = image.resize((new_width, target_height), PIL.Image.Resampling.LANCZOS)
    return image

def image_to_base64(image):
    buffered = BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

filenames_input = sorted([f for f in os.listdir(upload_folder) if is_image_file(f)])
filenames_output = sorted([f for f in os.listdir(result_folder) if is_image_file(f)])

if not filenames_input or not filenames_output:
    print(f'Error: no images found in {upload_folder} or {result_folder}.')
else:
    for filename, filename_output in zip(filenames_input, filenames_output):
        try:
            image_before = PIL.Image.open(os.path.join(upload_folder, filename))
            image_after = PIL.Image.open(os.path.join(result_folder, filename_output))

            max_width = min(image_after.size[0], 1000)
            image_after = resize_image_maintain_aspect(image_after, max_width)
            target_height = image_after.size[1]
            image_before = resize_image_maintain_aspect(image_before, max_width, target_height)

            if image_before.mode != 'RGB':
                image_before = image_before.convert('RGB')
            if image_after.mode != 'RGB':
                image_after = image_after.convert('RGB')

            before_base64 = image_to_base64(image_before)
            after_base64 = image_to_base64(image_after)

            html_code = f"""
            <div style="position: relative; width: {image_after.size[0]}px; height: {image_after.size[1]}px; margin-bottom: 20px;">
                <div style="position: relative; width: 100%; height: 100%; overflow: hidden;">
                    <img src="data:image/png;base64,{before_base64}" style="position: absolute; width: 100%; height: 100%;">
                    <div style="position: absolute; width: 100%; height: 100%; overflow: hidden; clip-path: inset(0 0 0 50%);">
                        <img src="data:image/png;base64,{after_base64}" style="position: absolute; width: 100%; height: 100%;">
                    </div>
                </div>
                <div class="slider" style="position: absolute; top: 0; bottom: 0; width: 4px; background: white; cursor: ew-resize; left: 50%; box-shadow: 0 0 5px rgba(0,0,0,0.5);">
                    <div style="position: absolute; top: 50%; transform: translateY(-50%); width: 20px; height: 20px; background: white; border-radius: 50%; left: -8px;"></div>
                </div>
                <div style="position: absolute; top: 8px; left: 8px; color: white; font-family: sans-serif; font-size: 14px; text-shadow: 0 1px 3px rgba(0,0,0,0.8);">Before</div>
                <div style="position: absolute; top: 8px; right: 8px; color: white; font-family: sans-serif; font-size: 14px; text-shadow: 0 1px 3px rgba(0,0,0,0.8);">After</div>
            </div>
            <script>
                document.querySelectorAll('.slider').forEach(slider => {{
                    let isDragging = false;
                    const container = slider.parentElement.querySelector('div:nth-child(1)');
                    const clipDiv = container.querySelector('div');

                    slider.addEventListener('mousedown', (e) => {{ isDragging = true; e.preventDefault(); }});
                    document.addEventListener('mouseup', () => {{ isDragging = false; }});
                    document.addEventListener('mousemove', (e) => {{
                        if (!isDragging) return;
                        const rect = container.getBoundingClientRect();
                        let x = e.clientX - rect.left;
                        if (x < 0) x = 0;
                        if (x > rect.width) x = rect.width;
                        const percentage = (x / rect.width) * 100;
                        slider.style.left = percentage + '%';
                        clipDiv.style.clipPath = `inset(0 0 0 ${{percentage}}%)`;
                    }});

                    slider.addEventListener('touchstart', (e) => {{ isDragging = true; e.preventDefault(); }});
                    document.addEventListener('touchend', () => {{ isDragging = false; }});
                    document.addEventListener('touchmove', (e) => {{
                        if (!isDragging) return;
                        const rect = container.getBoundingClientRect();
                        let x = e.touches[0].clientX - rect.left;
                        if (x < 0) x = 0;
                        if (x > rect.width) x = rect.width;
                        const percentage = (x / rect.width) * 100;
                        slider.style.left = percentage + '%';
                        clipDiv.style.clipPath = `inset(0 0 0 ${{percentage}}%)`;
                    }});
                }});
            </script>
            """

            display(HTML(html_code))

        except Exception as e:
            print(f'Error processing {filename} and {filename_output}: {e}')

In [ ]:
#@title ##**Download Results** { display-mode: "form" }
import os
from google.colab import files

zip_filename = 'MambaIRv2_result.zip'
if os.path.exists(zip_filename):
    os.remove(zip_filename)

os.system(f"zip -r -j {zip_filename} {result_folder}/*")
files.download(zip_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>